# Predicting Smartphone Addiction (Playground Series S6E8): Hill Climbing Ensemble（学習用解説付き写し）

- **コンペ**: [Predicting Smartphone Addiction](https://www.kaggle.com/competitions/playground-series-s6e8) — Playground Series Season 6 Episode 8。スマートフォン依存度（`addicted_label`、0〜1の連続的なリスクスコアとして扱われる）を予測するタスク。
- **元notebook**: [Hill Climbing Ensemble for Smartphone Addiction](https://www.kaggle.com/code/omidbaghchehsaraei/hill-climbing-ensemble-for-smartphone-addiction) by **Omid Baghcheh Saraei**（19 upvotes, Bronze, Public/Best Score 0.97005 V4）
- **手法の概要**: XGBoost・LightGBM・CatBoost・TabM・TabNet・FT-Transformer・ResNetなど複数の異なるモデル（著者の別notebook群で個別に学習・提出済み）のOOF（Out-of-Fold）予測とテスト予測を読み込み、**Hill Climbing（山登り法）** で貪欲にモデルを1つずつ追加しながら最適な重み付き線形結合を探索することで、単体最強モデルより高いスコアのアンサンブルを作る。
- **このノートブックについて**: 学習目的の解説付き写しであり、未実行（出力結果は含みません）。コード自体は元notebookの内容をほぼそのまま保持しています。


## 評価指標について（簡潔に）

Playground Series S6E8は Balanced Accuracy（PLAYBOOKに基づき今回は簡潔な扱いとしますが、この notebook では内部的にAUC最適化を行っています）ではなく、このnotebook内では `roc_auc_score` を評価指標として直接使っています。ROC-AUCは分類の閾値に依存せず「正例を負例より高くスコアリングできる確率」を測る指標で、依存度のようなリスクスコアのランキング精度を評価するのに適しています。Hill Climbingの目的関数もこの `roc_auc_score` そのものに設定されており、代理指標（loglossなど）を経由せず本番の評価指標を直接最適化している点が特徴です。


## Imports and Configuration

**何をしているか（What）**: 数値計算・データ処理ライブラリに加え、`hillclimbers` という専用パッケージから `climb_hill` 関数をインポートしています。`TARGET` 変数に目的変数のカラム名 `addicted_label` を設定しています。

**なぜこの構成か（Why）**: Hill Climbingアンサンブルは自作することも可能ですが、`hillclimbers` のような既製パッケージを使うことで、進捗表示・グラフ描画・負の重み許容などの実装を自前で書かずに済みます。


In [ ]:
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from hillclimbers import climb_hill, partial

warnings.filterwarnings("ignore")
TARGET = 'addicted_label' 

## Loading Base Datasets

**何をしているか（What）**: コンペ公式の train / test / sample_submission の3つのCSVを読み込みます。

**なぜこれが必要か（Why）**: Hill Climbing自体はモデルのOOF予測とtest予測だけでアンサンブルの重みを決められますが、正解ラベル（`train_df`の目的変数）がなければ「どの組み合わせがAUCを上げるか」を評価できないため、trainデータの読み込みも必須です。


In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e8/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e8/test.csv")
submission_df = pd.read_csv("/kaggle/input/competitions/playground-series-s6e8/sample_submission.csv")

## Loading Model Predictions (OOF and Test)

**何をしているか（What）**: 著者が事前に個別学習・公開していた6種類のモデル（XGBoost, RealMLP, TabM, ResNet, TabNet, FT-Transformer）それぞれについて、OOF予測（`oof.csv`の`oof_pred`列）とtest予測（`submission.csv`の`addicted_label`列）を、Kaggle上の「他notebookのOutputを直接Inputとして読み込む」機能で取得しています。

**なぜOOF予測が必要か（Why）**: アンサンブルの重みを「trainデータに対する予測」で最適化する際、学習に使ったデータそのもので評価すると過学習した重みになってしまいます。OOF（Out-of-Fold）予測は「そのモデルの学習には使われなかった fold のデータに対する予測」なので、汎化性能を反映した重み最適化ができます。


In [ ]:
oof = {}
test_pred = {}

# RealMLP
oof["RealMLP"] = pd.read_csv("/kaggle/input/notebooks/omidbaghchehsaraei/realmlp-for-predicting-smartphone-addiction/oof.csv")["oof_pred"]
test_pred["RealMLP"] = pd.read_csv("/kaggle/input/notebooks/omidbaghchehsaraei/realmlp-for-predicting-smartphone-addiction/submission.csv")["addicted_label"]

# XGBoost
oof["XGBoost"] = pd.read_csv("/kaggle/input/notebooks/omidbaghchehsaraei/xgboost-for-predicting-smartphone-addiction/oof.csv")["oof_pred"]
test_pred["XGBoost"] = pd.read_csv("/kaggle/input/notebooks/omidbaghchehsaraei/xgboost-for-predicting-smartphone-addiction/submission.csv")["addicted_label"]

# TabM
oof["TabM"] = pd.read_csv("/kaggle/input/notebooks/omidbaghchehsaraei/tabm-for-predicting-smartphone-addiction/oof.csv")["oof_pred"]
test_pred["TabM"] = pd.read_csv("/kaggle/input/notebooks/omidbaghchehsaraei/tabm-for-predicting-smartphone-addiction/submission.csv")["addicted_label"]

# ResNet
oof["ResNet"] = pd.read_csv("/kaggle/input/notebooks/omidbaghchehsaraei/resnet-for-predicting-smartphone-addiction/oof.csv")["oof_pred"]
test_pred["ResNet"] = pd.read_csv("/kaggle/input/notebooks/omidbaghchehsaraei/resnet-for-predicting-smartphone-addiction/submission.csv")["addicted_label"]

# TabNet
oof["TabNet"] = pd.read_csv("/kaggle/input/notebooks/omidbaghchehsaraei/tabnet-for-predicting-smartphone-addiction/oof.csv")["oof_pred"]
test_pred["TabNet"] = pd.read_csv("/kaggle/input/notebooks/omidbaghchehsaraei/tabnet-for-predicting-smartphone-addiction/submission.csv")["addicted_label"]

# FT-Transformer
oof["FT-Transformer"] = pd.read_csv("/kaggle/input/notebooks/omidbaghchehsaraei/ft-transformer-for-predicting-smartphone-addiction/oof.csv")["oof_pred"]
test_pred["FT-Transformer"] = pd.read_csv("/kaggle/input/notebooks/omidbaghchehsaraei/ft-transformer-for-predicting-smartphone-addiction/submission.csv")["addicted_label"]

# Convert to DataFrames
oof = pd.DataFrame(oof)
test_pred = pd.DataFrame(test_pred)

**補足（Why 6モデルなのか）**: 勾配ブースティング系（XGBoost）、深層学習の表形式データ用アーキテクチャ（TabM, TabNet, FT-Transformer, RealMLP, ResNet）と、多様なモデルファミリーを揃えています。アンサンブルは個々のモデルの予測誤差の相関が低いほど効果が出やすいため、単に精度が高いモデルを集めるだけでなく、異なる帰納バイアス（決定木ベース vs ニューラルネット系の複数アーキテクチャ）を持つモデルを混ぜるのが定石です。


## Hill Climbing Ensemble Optimization

**何をしているか（What）**: `climb_hill` 関数に、正解ラベルを含む `train_df`、目的変数名、最適化方向（`maximize`）、評価指標（`roc_auc_score`）、各モデルのOOF予測・test予測のDataFrameを渡し、貪欲法でアンサンブルの重みを探索させています。

**Hill Climbingのアルゴリズム（What/Why）**: 単純なアイデアで、(1) 最初は最もスコアが良い単体モデルからスタートし、(2) 残りの各モデルについて「今のアンサンブルに少し混ぜたらスコアが上がるか」を試し、(3) 最もスコアが上がる重みでそのモデルを採用、(4) スコアが改善しなくなるまで(2)-(3)を繰り返す、という貪欲法です。全モデルの重みを同時に最適化する凸最適化と比べると理論的な最適性は劣りますが、実装が単純で局所解に陥りにくく、OOFに対する過学習も比較的抑えられるため、Kaggleのアンサンブル手法として広く使われています。

**`negative_weights=True` の意味（Why）**: 通常のアンサンブルは重みを正の範囲に制限しますが、これを許可すると「あるモデルの予測を引き算する」ことも探索対象になります。ログ出力を見ると実際に ResNet の重みが `-0.042` と負になっており、ResNetの予測を混ぜるのではなく打ち消す方向に使うことでスコアが改善したことが分かります。


In [ ]:
hc_test, hc_oof = climb_hill(
    train=train_df,
    target=TARGET,
    objective='maximize',
    eval_metric=partial(roc_auc_score),
    oof_pred_df=oof,
    test_pred_df=test_pred,
    plot_hill=True,
    plot_hist=False,
    precision=0.001,
    negative_weights=True,
    return_oof_preds=True,
)

**実行結果（著者のログより、要約）**:

```
Models to be ensembled | (6 total):
RealMLP:        0.96844 (best solo model)
TabNet:         0.96756
TabM:           0.96751
XGBoost:        0.96681
FT-Transformer: 0.96657
ResNet:         0.96610

Iteration: 1 | Model added: TabNet          | Best weight: 0.265  | Best roc_auc_score: 0.96857
Iteration: 2 | Model added: XGBoost         | Best weight: 0.083  | Best roc_auc_score: 0.96859
Iteration: 3 | Model added: TabM            | Best weight: 0.074  | Best roc_auc_score: 0.96859
Iteration: 4 | Model added: ResNet          | Best weight: -0.042 | Best roc_auc_score: 0.96860
Iteration: 5 | Model added: FT-Transformer  | Best weight: 0.011  | Best roc_auc_score: 0.96860
```

**この結果から読み取れること（What/Why）**: 単体最強のRealMLP（AUC 0.96844）に対し、他モデルを段階的に混ぜることで最終的にAUC 0.96860まで改善しています。改善幅は小さく見えますが、Kaggleのリーダーボードのように僅差で順位が決まる場では意味のある差です。特に興味深いのは、単体では最下位（0.96610）だったResNetが、負の重み（-0.042）としてアンサンブルに寄与している点です。これはResNetの予測誤差のパターンが他モデルと系統的に異なり、その誤差を「打ち消す」方向に使うことでアンサンブル全体の精度が上がったことを示唆します。


## Submission

**何をしているか（What）**: `climb_hill` が返した最適化済みのtest予測（`hc_test`）から `submission.csv` を組み立てて保存します（コード自体は著者によって折りたたまれていましたが、出力は以下のような形式です）。

```
   id       addicted_label
0  691369   0.999765
1  691370   0.968007
2  691371   0.820711
3  691372   0.996117
4  691373   0.999196
```


## この手法から学べる主要テクニック

- 複数の異なるモデルファミリー（勾配ブースティング木＋複数の深層学習表形式アーキテクチャ）のOOF予測を集めてアンサンブルする、Kaggleで広く使われる王道の手法。
- Hill Climbing（貪欲法によるモデル逐次追加＋重み探索）は、全モデル同時最適化より単純で過学習しにくいアンサンブル構築法。
- アンサンブルの重みに負の値を許容することで、「予測を打ち消す」形の寄与を捉えられる。
- 本番の評価指標（ここではROC-AUC）をアンサンブル最適化の目的関数に直接使うことで、代理指標経由の間接的な最適化より確実にスコアへ寄与する組み合わせを選べる。
- オンライン/逐次学習の文脈と同様、**評価・検証に使うウィンドウの大きさが手法の前提と整合しているか**を常に意識する必要がある（この点は同日扱った実コンペのnotebookでも重要な論点として登場する）。
